# Modul 9: Multi-Agent Systeme & A2A

**Agent Systems: Vom Konzept zum eigenen Orchestrator**

In diesem Notebook baust du ein 2-Agent-System mit Orchestrator und Worker.

🎯 **Lernziel:** Message-Passing zwischen Agents implementieren und Task-Delegation verstehen.

---

## Multi-Agent = Spezialisierung + Koordination

```
          ┌──────────────┐
          │ ORCHESTRATOR  │  ← Plant, delegiert, konsolidiert
          └──────┬───────┘
                 │
        ┌────────┼────────┐
        ▼        ▼        ▼
   ┌─────────┐ ┌─────┐ ┌─────────┐
   │ Worker  │ │ W2  │ │ Worker  │  ← Spezialisiert
   │Research │ │Code │ │ Review  │
   └─────────┘ └─────┘ └─────────┘
```

Ng (2024) definiert Multi-Agent als **Pattern 4/4**: Spezialisierte Agents für Teilaufgaben.

**A2A (Agent-to-Agent Protocol, Google 2025):** Standard für Agent-Kommunikation — *"A2A ist für Agents, was HTTP für Webserver ist."*

In [ ]:
# 2-Agent-System: Orchestrator + Worker

import time
from datetime import datetime


class Message:
    """Nachricht zwischen Agents."""
    def __init__(self, sender, receiver, task, payload=None):
        self.sender = sender
        self.receiver = receiver
        self.task = task
        self.payload = payload
        self.timestamp = datetime.now().isoformat()
        self.status = 'pending'

    def __repr__(self):
        return f'Message({self.sender} -> {self.receiver}: "{self.task}" [{self.status}])'


class WorkerAgent:
    """Spezialisierter Worker-Agent."""
    def __init__(self, name, specialty, process_fn):
        self.name = name
        self.specialty = specialty
        self.process_fn = process_fn
        self.inbox = []
        self.completed = []

    def receive(self, message):
        self.inbox.append(message)
        print(f'  📨 {self.name} empfaengt: "{message.task}"')

    def work(self):
        """Verarbeitet alle Nachrichten in der Inbox."""
        results = []
        while self.inbox:
            msg = self.inbox.pop(0)
            print(f'  ⚙️  {self.name} arbeitet an: "{msg.task}"')
            result = self.process_fn(msg.task)
            msg.status = 'completed'
            msg.payload = result
            self.completed.append(msg)
            results.append(msg)
        return results


class OrchestratorAgent:
    """Koordiniert Worker-Agents."""
    def __init__(self, name):
        self.name = name
        self.workers = {}
        self.task_log = []

    def register_worker(self, worker):
        self.workers[worker.specialty] = worker
        print(f'  Worker registriert: {worker.name} ({worker.specialty})')

    def delegate(self, task, specialty):
        """Delegiert eine Aufgabe an den passenden Worker."""
        worker = self.workers.get(specialty)
        if not worker:
            print(f'  ❌ Kein Worker fuer "{specialty}"!')
            return None

        msg = Message(self.name, worker.name, task)
        self.task_log.append(msg)
        print(f'\n📋 {self.name} delegiert an {worker.name}:')
        worker.receive(msg)
        return msg

    def collect_results(self):
        """Sammelt Ergebnisse von allen Workern."""
        all_results = []
        for worker in self.workers.values():
            results = worker.work()
            all_results.extend(results)
        return all_results

    def synthesize(self, results):
        """Fasst Ergebnisse zusammen."""
        print(f'\n📊 {self.name} konsolidiert {len(results)} Ergebnisse:')
        synthesis = []
        for r in results:
            print(f'  ✅ {r.receiver}: {r.payload}')
            synthesis.append(r.payload)
        summary = ' | '.join(synthesis)
        print(f'\n💡 Synthese: {summary}')
        return summary


# === Demo: Research-Pipeline ===
print('=== Multi-Agent Research Pipeline ===\n')

# Orchestrator erstellen
orchestrator = OrchestratorAgent('Orchestrator')

# Worker erstellen
research_worker = WorkerAgent(
    'ResearchBot', 'research',
    lambda task: f'3 Quellen zu "{task}" gefunden und zusammengefasst'
)

review_worker = WorkerAgent(
    'ReviewBot', 'review',
    lambda task: f'Qualitaetscheck fuer "{task}": 8/10 Punkte, 1 Verbesserung'
)

orchestrator.register_worker(research_worker)
orchestrator.register_worker(review_worker)

# Aufgaben delegieren
orchestrator.delegate('Agent-Architekturen 2025', 'research')
orchestrator.delegate('Research-Ergebnis pruefen', 'review')

# Ergebnisse sammeln und zusammenfassen
results = orchestrator.collect_results()
orchestrator.synthesize(results)

In [ ]:
# Task-Delegation mit Routing

class SmartOrchestrator(OrchestratorAgent):
    """Orchestrator mit automatischem Routing."""

    def __init__(self, name):
        super().__init__(name)
        self.routing_rules = {}  # keyword -> specialty

    def add_route(self, keyword, specialty):
        self.routing_rules[keyword] = specialty

    def auto_delegate(self, task):
        """Erkennt automatisch den passenden Worker."""
        task_lower = task.lower()
        for keyword, specialty in self.routing_rules.items():
            if keyword in task_lower:
                return self.delegate(task, specialty)
        print(f'  ⚠️ Kein Routing fuer: "{task}" — Orchestrator uebernimmt selbst.')
        return None

    def decompose_and_run(self, complex_task):
        """Zerlegt eine komplexe Aufgabe in Sub-Tasks."""
        print(f'\n🧩 Aufgabe zerlegen: "{complex_task}"')
        # Einfache Zerlegung: Komma-getrennte Sub-Tasks
        subtasks = [t.strip() for t in complex_task.split(',')]
        print(f'   → {len(subtasks)} Sub-Tasks erkannt')

        for subtask in subtasks:
            self.auto_delegate(subtask)

        results = self.collect_results()
        return self.synthesize(results)


# Demo
print('=== Smart Orchestrator ===\n')

smart = SmartOrchestrator('SmartBot')
smart.register_worker(WorkerAgent('Researcher', 'research',
    lambda t: f'Research zu "{t}": 5 Paper gefunden'))
smart.register_worker(WorkerAgent('Coder', 'code',
    lambda t: f'Code fuer "{t}": 42 Zeilen Python'))
smart.register_worker(WorkerAgent('Reviewer', 'review',
    lambda t: f'Review von "{t}": Approved mit 2 Anmerkungen'))

smart.add_route('recherchiere', 'research')
smart.add_route('implementiere', 'code')
smart.add_route('pruefe', 'review')

# Komplexe Aufgabe zerlegen und ausfuehren
smart.decompose_and_run(
    'Recherchiere Agent-Patterns, Implementiere einen Prototyp, Pruefe das Ergebnis'
)

In [ ]:
# 🎯 ÜBUNG: Füge einen dritten Worker hinzu
#
# Aufgabe:
# 1. Erstelle einen 'summary'-Worker der Ergebnisse zusammenfasst
# 2. Registriere ihn beim SmartOrchestrator
# 3. Füge eine Routing-Regel hinzu (Keyword: 'fasse zusammen')
# 4. Teste mit einer 4-teiligen Aufgabe

# TODO: Summary-Worker erstellen
# summary_worker = WorkerAgent('Summarizer', 'summary', lambda t: ...)

# TODO: Registrieren
# smart.register_worker(summary_worker)

# TODO: Route hinzufuegen
# smart.add_route('fasse zusammen', 'summary')

# TODO: Testen
# smart.decompose_and_run(
#     'Recherchiere AI Trends, Implementiere Demo, Pruefe Code, Fasse zusammen'
# )

In [ ]:
# ✅ LÖSUNG

summary_worker = WorkerAgent('Summarizer', 'summary',
    lambda t: f'Zusammenfassung: Alle Ergebnisse konsolidiert, 3 Key Takeaways erstellt'
)

smart.register_worker(summary_worker)
smart.add_route('fasse zusammen', 'summary')

print('=== 4-Schritt Pipeline ===')
smart.decompose_and_run(
    'Recherchiere AI Trends, Implementiere Demo, Pruefe Code, Fasse zusammen'
)

## Orchestration Patterns

| Pattern | Fluss | Einsatz |
|---------|-------|---------|
| **Sequential** | A → B → C | Pipeline, Review-Kette |
| **Concurrent** | A ∥ B ∥ C → Merge | Parallele Recherche |
| **Handoff** | A → B (mit Kontext) | Spezialist übernimmt |
| **Orchestrator-Worker** | O → W1, W2, W3 → O | Komplexe Aufgaben |

---

### ✅ Self-Check
- [ ] Ich kann den Unterschied zwischen Orchestrator und Worker erklären
- [ ] Ich verstehe Message-Passing und Task-Delegation
- [ ] Ich kann die 4 Orchestration Patterns benennen

**→ Weiter zu [Modul 10: Automatisierung](https://colab.research.google.com/github/planck-lab/agent-systems/blob/main/notebooks/modul10_automatisierung.ipynb)**